# Sensitivity analysis of the trivariate joint contact force emulator

Benemerito et al. (2022) rank the 29 muscle strengths $F_{\max}$ of a musculoskeletal model by their influence on the hip, knee and ankle joint contact forces (JCF). They draw the strengths independently from normal distributions, run OpenSim 200 times, fit a Gaussian process (GP) emulator to the runs, and compute Sobol indices at every point of the gait cycle from 30,720 emulator predictions.

Our data are different. The muscle strengths are fixed, and what changes from one stride to the next is the walking itself: 480 strides of 12 participants at several belt speeds. The inputs of the emulator are eight measured curves over the gait cycle (three ground reaction force components and five joint angles) and three scalars (belt speed, body mass, height). These inputs are strongly dependent. A faster stride has a larger vertical force, more knee flexion and more hip flexion at the same time, so the inputs cannot be redrawn one at a time as in a Sobol design: the emulator would be asked to predict strides that nobody walks.

This notebook therefore replaces the Sobol indices with **Shapley effects** (Owen, 2014; Song, Nelson and Staum, 2016), which remain meaningful for dependent inputs, and replaces the independent sampling with a **joint model of the inputs** built on functional principal components. The question is the same as in the paper: at each percent of the gait cycle, which input explains how much of the stride-to-stride variance of each joint contact force?

The eleven inputs are treated as eleven **groups**: each curve is one group (all its 101 values together), and each scalar is a group of its own.

### What you have to do

The data loading, the emulator and all the figures are written for you. The cells marked **TODO** are yours. Each TODO cell is followed by a check cell; run it, and do not move on until the check passes. The steps are:

1. Functional PCA of each input curve.
2. A multivariate normal model of the principal component scores and the three scalars.
3. Conditional sampling from that model.
4. Turning sampled scores back into emulator inputs, and checking that the sampled strides look like real strides.
5. The cost function $c(u)$ for a set of groups $u$.
6. Shapley effects from the costs, tested first on a toy problem with a known answer.
7. Variance-renormalized effects and the figures.

Use a GPU runtime if one is available (Runtime > Change runtime type). The notebook also runs on a CPU, more slowly.

## Setup

In [ ]:
import itertools, math, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import minimize

torch.set_default_dtype(torch.float64)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
rng = np.random.default_rng(2026)

## Data

One row of the data files is one stride of the right leg, from heel strike to the next heel strike. Row $i$ of `x-*.csv` and row $i$ of `y-*.csv` are the same stride. There are 480 training strides from twelve participants and 140 test strides from three other participants. The file `data/log.txt` describes every column.

A curve called `name` occupies the 101 columns `name_t000` to `name_t100`, one per percent of the gait cycle. Stance is roughly the first 62 % of the cycle and swing the rest. The input curves are

| group | columns | unit | source |
|---|---|---|---|
| `grf_vertical_BW` | vertical ground reaction force | body weights | force plates |
| `grf_anteroposterior_BW` | fore-aft ground reaction force | body weights | force plates |
| `grf_mediolateral_BW` | side-to-side ground reaction force | body weights | force plates |
| `hip_flexion_deg` | hip flexion angle | degrees | OpenSim inverse kinematics |
| `hip_adduction_deg` | hip adduction angle | degrees | inverse kinematics |
| `hip_rotation_deg` | hip rotation angle | degrees | inverse kinematics |
| `knee_flexion_deg` | knee flexion angle | degrees | inverse kinematics |
| `ankle_angle_deg` | ankle dorsiflexion angle | degrees | inverse kinematics |

and the scalars are `belt_speed_mps`, `mass_kg` and `height_m`. The outputs are the curves `hip_jcf_BW`, `knee_jcf_BW` and `ankle_jcf_BW`, computed by the OpenSim joint reaction analysis.

In [ ]:
if not os.path.exists("data/x-train.csv"):
    !gdown --folder "https://drive.google.com/drive/folders/1CI5OAxzB9qYI9vIxW1f0cGE4SN1hhyBj" -q
x_train = pd.read_csv("data/x-train.csv")
y_train = pd.read_csv("data/y-train.csv")
x_test = pd.read_csv("data/x-test.csv")
y_test = pd.read_csv("data/y-test.csv")
x_train.shape, y_train.shape, x_test.shape, y_test.shape

In [ ]:
GRID = np.arange(101)
INPUT_CURVES = ["grf_vertical_BW", "grf_anteroposterior_BW", "grf_mediolateral_BW",
                "hip_flexion_deg", "hip_adduction_deg", "hip_rotation_deg", "knee_flexion_deg", "ankle_angle_deg"]
SCALARS = ["belt_speed_mps", "mass_kg", "height_m"]
GROUP_NAMES = INPUT_CURVES + SCALARS                     # the eleven input groups, in this order everywhere
JOINTS = ["hip", "knee", "ankle"]

def curve(df, name):
    # the curve `name` of every stride, shape (strides, 101)
    return df[[f"{name}_t{t:03d}" for t in GRID]].to_numpy()

## The emulator

The emulator is the trivariate Matérn model of `jcf-gp-trivariate-colab.ipynb`. The input of a stride is the vector of its eight curves and three scalars, standardized with the training strides (811 numbers). Each joint force is standardized at each gait point, and the 101 gait points are treated as replicates of one trivariate Gaussian process over the inputs, so one set of eleven parameters serves the whole cycle. The original notebook fits it with the mixed-precision Cholesky factorization. Here the covariance matrix is only $1{,}440 \times 1{,}440$, so an ordinary double-precision Cholesky factorization is used. The fit takes a minute or two.

You do not need to change anything in this section. What you need from it is the function `emulate(F)`, which takes a matrix of standardized inputs with one row per stride and returns the predicted JCF curves of the three joints, shape `(3, strides, 101)`, in body weights. `features(curves, scalars)` builds those standardized inputs from raw curves and scalars.

In [ ]:
CURVE_MEAN = {c: curve(x_train, c).mean(axis=0) for c in INPUT_CURVES}     # mean curve, (101,)
CURVE_SD = {c: curve(x_train, c).std() for c in INPUT_CURVES}             # one standard deviation per curve
SCALAR_MEAN = {s: x_train[s].mean() for s in SCALARS}
SCALAR_SD = {s: x_train[s].std() for s in SCALARS}

def features(curves, scalars):
    # curves: dict name -> (strides, 101) array in the original units; scalars: (strides, 3) array (speed, mass, height)
    cols = [(curves[c] - CURVE_MEAN[c]) / CURVE_SD[c] for c in INPUT_CURVES]
    cols += [((scalars[:, k] - SCALAR_MEAN[s]) / SCALAR_SD[s])[:, None] for k, s in enumerate(SCALARS)]
    return np.hstack(cols)                                               # (strides, 811)

def features_of(df):
    return features({c: curve(df, c) for c in INPUT_CURVES}, df[SCALARS].to_numpy())

F_train, F_test = features_of(x_train), features_of(x_test)
Y_train = np.stack([curve(y_train, f"{j}_jcf_BW") for j in JOINTS])        # (3, 480, 101)
Y_test = np.stack([curve(y_test, f"{j}_jcf_BW") for j in JOINTS])
mean_t, scale_t = Y_train.mean(axis=1), Y_train.std(axis=1)
Z_train = (Y_train - mean_t[:, None, :]) / scale_t[:, None, :]
n, D = F_train.shape
F_train.shape

In [ ]:
def matern(d):
    s = 5.0**0.5 * d
    return (1.0 + s + s * s / 3.0) * torch.exp(-s)

def correlation_matrix(raw):
    a = (np.pi / 2) * np.tanh(np.asarray(raw, dtype=float))
    L = np.array([[1.0, 0.0, 0.0], [np.sin(a[0]), np.cos(a[0]), 0.0],
                  [np.sin(a[1]), np.cos(a[1]) * np.sin(a[2]), np.cos(a[1]) * np.cos(a[2])]])
    return L @ L.T

def unpack(theta):
    # theta = log input ranges (3), correlation angles (3), log variance ratios of knee and ankle (2), log relative nugget effects (3)
    l = np.exp(theta[0:3]); beta = correlation_matrix(theta[3:6])
    v = np.concatenate([[1.0], np.exp(theta[6:8])]); eta = np.exp(theta[8:11])
    a2 = 1.0 / l**2
    inverse_l = np.sqrt((a2[:, None] + a2[None, :]) / 2)
    gamma = (np.sqrt(a2[:, None] * a2[None, :]) / inverse_l**2) ** 2.5
    rho = beta * gamma
    return dict(l=l, v=v, eta=eta, rho=rho, amplitude=rho * np.sqrt(np.outer(v, v)), inverse_l=inverse_l)

Ft = torch.tensor(F_train, device=DEVICE)
H = torch.cdist(Ft, Ft) / D**0.5                                      # root mean squared input difference between strides
y_rep = torch.tensor(Z_train.reshape(3 * n, 101), device=DEVICE)       # one column per gait point
joint = torch.arange(3 * n, device=DEVICE) // n
stride = torch.arange(3 * n, device=DEVICE) % n

def covariance(p):
    amp = torch.tensor(p["amplitude"], device=DEVICE); inv = torch.tensor(p["inverse_l"], device=DEVICE)
    K = amp[joint][:, joint] * matern(H[stride][:, stride] * inv[joint][:, joint])
    return K + torch.diag(torch.tensor(p["eta"], device=DEVICE)[joint])

def negative_log_likelihood(theta):
    try:
        L = torch.linalg.cholesky(covariance(unpack(theta)))
    except Exception:
        return 1e10
    alpha = torch.cholesky_solve(y_rep, L)
    N, T = 3 * n, 101
    sigma2 = float((y_rep * alpha).sum()) / (N * T)                   # profiled variance
    return 0.5 * (N * T * np.log(sigma2) + T * 2 * float(torch.log(torch.diagonal(L)).sum()) + N * T)

t0 = time.time()
theta0 = np.concatenate([np.log([0.354, 0.287, 0.313]), [0.3, 0.3, 0.3], [0.0, 0.0], np.log([0.01] * 3)])
bounds = [(np.log(0.05), np.log(5.0))] * 3 + [(-2.0, 2.0)] * 3 + [(-3.0, 3.0)] * 2 + [(np.log(1e-5), np.log(0.5))] * 3
fit = minimize(negative_log_likelihood, theta0, method="L-BFGS-B", bounds=bounds, options=dict(maxiter=60))
P = unpack(fit.x)
print(f"fitted in {time.time() - t0:.0f} s, negative log likelihood {fit.fun:,.1f}")
print("input ranges (hip, knee, ankle)", P["l"].round(3))
pd.DataFrame(P["rho"], index=JOINTS, columns=JOINTS).round(3)

In [ ]:
L_fit = torch.linalg.cholesky(covariance(P))
ALPHA = torch.cholesky_solve(y_rep, L_fit)                            # (3n, 101)
AMP = torch.tensor(P["amplitude"], device=DEVICE)
INV = torch.tensor(P["inverse_l"], device=DEVICE)

def emulate(F_new, batch=4000):
    # predictive mean of the JCF curves of new strides, (3 joints, strides, 101 gait points), in BW
    out = []
    for a in range(0, len(F_new), batch):
        Fn = torch.tensor(F_new[a:a + batch], device=DEVICE)
        h = torch.cdist(Fn, Ft) / D**0.5
        z = torch.stack([(AMP[j][joint] * matern(h[:, stride] * INV[j][joint])) @ ALPHA for j in range(3)])
        out.append(z.cpu().numpy())
    return mean_t[:, None, :] + scale_t[:, None, :] * np.concatenate(out, axis=1)

pred_test = emulate(F_test)
pd.Series(np.sqrt(((pred_test - Y_test)**2).mean(axis=(1, 2))), index=JOINTS, name="test RMSE [BW]").round(3)

The test RMSE should be roughly 0.2 to 0.4 BW. If it is far larger, something went wrong above; ask before continuing.

## Step 1. Functional PCA of each input curve

Each input curve has 101 strongly correlated values. To build a distribution over curves, summarize every curve by a few principal component scores. For curve $c$, let $\mathbf{C}_c$ be the $480 \times 101$ matrix of its training values and $\bar{\mathbf{c}}$ its mean curve (`CURVE_MEAN[c]`). The singular value decomposition of the centered matrix $\mathbf{C}_c - \mathbf{1}\bar{\mathbf{c}}^\top = \mathbf{U}\mathbf{S}\mathbf{V}^\top$ gives the principal directions (rows of $\mathbf{V}^\top$). Keep the first $K_c$ rows, where $K_c$ is the smallest number whose cumulative share of $\sum s_k^2$ reaches 95 %. The scores of a stride are its centered curve times the kept directions, and the curve is approximately rebuilt as $\bar{\mathbf{c}} + \text{scores} \cdot \mathbf{V}_{K_c}^\top$.

**TODO.** Fill the dictionary `PCS` so that `PCS[c]` is the $K_c \times 101$ array of kept directions of curve `c`.

In [ ]:
VARIANCE_KEPT = 0.95
PCS = {}
for c in INPUT_CURVES:
    # TODO: center curve(x_train, c) with CURVE_MEAN[c], take the SVD, choose K_c, store the first K_c rows of V^T
    raise NotImplementedError

**Check.** The table shows $K_c$ and the reconstruction error of every curve. Expect a handful of components per curve and errors that are small next to the size of the curves.

In [ ]:
rows = []
for c in INPUT_CURVES:
    C = curve(x_train, c)
    S = (C - CURVE_MEAN[c]) @ PCS[c].T
    R = CURVE_MEAN[c] + S @ PCS[c]
    assert PCS[c].shape[1] == 101 and np.allclose(PCS[c] @ PCS[c].T, np.eye(len(PCS[c])), atol=1e-8), c
    rows.append({"curve": c, "K": len(PCS[c]), "RMS reconstruction error": np.sqrt(((C - R)**2).mean()), "RMS of the curve": np.sqrt((C**2).mean())})
pd.DataFrame(rows).set_index("curve").round(3)

## Step 2. A joint model of the inputs

Put the scores of the eight curves and the three scalars side by side in one vector per stride, in the order of `GROUP_NAMES`. This vector has $p = \sum_c K_c + 3$ entries. Model it as multivariate normal, $\mathbf{u} \sim \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$, with $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$ the sample mean and covariance of the 480 training strides. The covariance is what keeps the dependence between the inputs: through it, a large vertical force comes with a fast speed and a large knee flexion.

**TODO.** Build

- `U_train`, the $480 \times p$ matrix of the vectors of the training strides;
- `GROUPS`, a list of eleven lists, where `GROUPS[g]` holds the column indices of group `g` in `U_train`;
- `MU` and `SIGMA`.

Keep the three scalars in their original units (m/s, kg, m).

In [ ]:
# TODO
U_train = None
GROUPS = None
MU, SIGMA = None, None

**Check.**

In [ ]:
p = U_train.shape[1]
assert U_train.shape == (480, p) and len(GROUPS) == 11
assert sorted(sum(GROUPS, [])) == list(range(p)), "every column must belong to exactly one group"
assert MU.shape == (p,) and SIGMA.shape == (p, p) and np.all(np.linalg.eigvalsh(SIGMA) > 0)
print(f"p = {p}")
corr = SIGMA / np.sqrt(np.outer(np.diag(SIGMA), np.diag(SIGMA)))
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ticks = [g[0] for g in GROUPS]
ax.set_xticks(ticks); ax.set_xticklabels(GROUP_NAMES, rotation=90, fontsize=8)
ax.set_yticks(ticks); ax.set_yticklabels(GROUP_NAMES, fontsize=8)
ax.set_title("correlation between the input scores")
fig.colorbar(im); plt.show()

Look at the figure and write down two pairs of groups that are strongly correlated. They are the reason a Sobol design would not work here.

## Step 3. Conditional sampling

Shapley effects need draws of some groups **given** fixed values of the others. Split the entries of $\mathbf{u}$ into the known set $k$ and the unknown set $r$. For the multivariate normal,
$$
\mathbf{u}_r \mid \mathbf{u}_k \sim \mathcal{N}\!\Big(\boldsymbol{\mu}_r + \boldsymbol{\Sigma}_{rk}\boldsymbol{\Sigma}_{kk}^{-1}(\mathbf{u}_k - \boldsymbol{\mu}_k),\;
\boldsymbol{\Sigma}_{rr} - \boldsymbol{\Sigma}_{rk}\boldsymbol{\Sigma}_{kk}^{-1}\boldsymbol{\Sigma}_{kr}\Big).
$$
The conditional covariance does not depend on $\mathbf{u}_k$, so one Cholesky factor of it serves all rows.

**TODO.** Write `sample_conditional(known_values, known, unknown, n_draws)`. `known` and `unknown` are lists of column indices, `known_values` has shape `(m, len(known))`, and the function returns an array of shape `(m, n_draws, len(unknown))`. When `known` is empty, return draws from the marginal of the unknown entries. Use `np.linalg.solve`, not an explicit inverse, and the global `rng`.

In [ ]:
def sample_conditional(known_values, known, unknown, n_draws):
    # TODO
    raise NotImplementedError

**Check.** Many draws given one fixed known value must have the conditional mean and covariance of the formula.

In [ ]:
known, unknown = GROUPS[GROUP_NAMES.index("belt_speed_mps")], GROUPS[GROUP_NAMES.index("grf_vertical_BW")]
value = np.array([[1.5]])
draws = sample_conditional(value, known, unknown, 200000)[0]
W = SIGMA[np.ix_(unknown, known)] / SIGMA[known[0], known[0]]
assert draws.shape == (200000, len(unknown))
assert np.allclose(draws.mean(axis=0), MU[unknown] + (value[0] - MU[known]) @ W.T, atol=0.05 * np.sqrt(np.diag(SIGMA)[unknown]).max())
assert np.allclose(np.cov(draws, rowvar=False), SIGMA[np.ix_(unknown, unknown)] - W @ SIGMA[np.ix_(known, unknown)], rtol=0.05, atol=1e-3)
assert sample_conditional(np.zeros((4, 0)), [], unknown, 3).shape == (4, 3, len(unknown))
print("sample_conditional passes")

## Step 4. From scores back to emulator inputs

**TODO.** Write `to_features(U)`, which takes an array of score vectors of shape `(strides, p)`, rebuilds the eight curves as mean curve plus scores times directions, takes the three scalars from the last three columns, and returns `features(curves, scalars)`.

In [ ]:
def to_features(U):
    # TODO
    raise NotImplementedError

def to_curves(U):
    # optional helper for the plot below: dict name -> (strides, 101) rebuilt curves
    raise NotImplementedError

**Check.** First, the rebuilt training strides must give almost the same emulator predictions as the real ones; the loss is only the 5 % of curve variance dropped by the PCA. Then 300 strides drawn from the model are drawn over the 480 real training strides. **This is the most important check of the notebook.** If the simulated strides do not look like real walking, every number after this point is meaningless. Look in particular at swing, where the ground reaction forces are zero for real strides.

In [ ]:
F_rebuilt = to_features(U_train)
assert F_rebuilt.shape == F_train.shape
gap = np.sqrt(((emulate(F_rebuilt) - emulate(F_train))**2).mean(axis=(1, 2)))
print("RMS change of the predicted JCF when the curves are replaced by their PCA reconstruction [BW]:", dict(zip(JOINTS, gap.round(3))))

U_sim = rng.multivariate_normal(MU, SIGMA, size=300)
sim = to_curves(U_sim)
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, c in zip(axes.flat, INPUT_CURVES):
    ax.plot(GRID, curve(x_train, c).T, color="#1f3b5c", lw=0.5, alpha=0.15)
    ax.plot(GRID, sim[c].T, color="#D55E00", lw=0.5, alpha=0.2)
    ax.set_title(c, loc="left"); ax.set_xlim(0, 100); ax.grid(alpha=0.3)
axes[0][0].plot([], [], color="#1f3b5c", label="real"); axes[0][0].plot([], [], color="#D55E00", label="simulated")
axes[0][0].legend(frameon=False)
fig.tight_layout(); plt.show()

Write two or three sentences on where the simulated curves differ from the real ones and why (hint: a normal distribution cannot produce the exact zero of the forces in swing, and speed has only four values in the data).

## Step 5. The cost function

Let $Y = \mathrm{JCF}_j(t)$ be the emulated force of joint $j$ at gait point $t$, seen as a random variable through the random input $\mathbf{u}$. For a set $u$ of groups, with complement $-u$, define the cost
$$
c(u) = \mathbb{E}\big[\operatorname{Var}(Y \mid \mathbf{u}_{-u})\big],
$$
the variance that is left when every group outside $u$ is known. It satisfies $c(\emptyset) = 0$ and $c(\text{all groups}) = \operatorname{Var}(Y)$. Song, Nelson and Staum (2016) estimate it with a double loop:

1. draw $N_O$ outer vectors from $\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ and keep only their entries in $-u$;
2. for each outer vector draw $N_I$ inner values of the entries in $u$ from `sample_conditional`;
3. emulate all $N_O N_I$ strides, take the sample variance (with `ddof=1`) over the $N_I$ inner draws of each outer vector, and average over the outer vectors.

All three joints and all 101 gait points come out of the same predictions, so `cost(u)` returns an array of shape `(3, 101)`. Song et al. show that a small $N_I$ (2 or 3) with a larger $N_O$ is the efficient choice.

**TODO.** Write `cost(u, n_outer, n_inner)`, where `u` is a tuple of group numbers. Handle the two ends: return zeros for the empty set and `VAR_Y` for the full set.

In [ ]:
U_big = rng.multivariate_normal(MU, SIGMA, size=20000)
VAR_Y = emulate(to_features(U_big)).var(axis=1)              # Var(Y) of every joint and gait point, (3, 101)
d = len(GROUPS)

In [ ]:
def cost(u, n_outer=200, n_inner=3):
    # TODO
    raise NotImplementedError

**Check.** The cost grows with the set: leaving more groups unknown leaves more variance. With Monte Carlo noise this holds on average, not at every gait point.

In [ ]:
c_one = cost((GROUP_NAMES.index("grf_vertical_BW"),), 400)
c_all_but_one = cost(tuple(g for g in range(d) if g != GROUP_NAMES.index("height_m")), 400)
assert c_one.shape == (3, 101)
print("mean c(u)/Var(Y) for u = {vertical GRF}:       ", (c_one / VAR_Y).mean(axis=1).round(3))
print("mean c(u)/Var(Y) for u = all groups but height:", (c_all_but_one / VAR_Y).mean(axis=1).round(3))
assert np.all((c_one / VAR_Y).mean(axis=1) < (c_all_but_one / VAR_Y).mean(axis=1))

## Step 6. Shapley effects

With $d = 11$ groups, the Shapley effect of group $i$ is
$$
\mathrm{Sh}_i = \frac{1}{\operatorname{Var}(Y)} \sum_{u \subseteq \{1,\dots,d\} \setminus \{i\}} \frac{|u|!\,(d - |u| - 1)!}{d!}\,\big[c(u \cup \{i\}) - c(u)\big],
$$
the extra variance group $i$ brings, averaged over every order in which the groups can be added. The effects are nonnegative for this cost and sum to one, $\sum_i \mathrm{Sh}_i = 1$, also when the inputs are dependent; that is the property the Sobol indices lose. Correlated groups share the credit instead of each claiming all of it.

**TODO.** Write `shapley(costs, d)`, where `costs` is a dictionary from every sorted tuple $u$ (including the empty tuple and the full tuple) to its cost array. It returns an array of shape `(d,) + cost shape` with the effects, **not yet** divided by $\operatorname{Var}(Y)$.

In [ ]:
def shapley(costs, d):
    # TODO
    raise NotImplementedError

**Check on a toy problem.** Take $Y = \beta_1 X_1 + \beta_2 X_2$ with $X_1, X_2$ standard normal and correlation $\rho$. The costs are known exactly: $c(\{1\}) = \beta_1^2(1-\rho^2)$, $c(\{2\}) = \beta_2^2(1-\rho^2)$, and $c(\{1,2\}) = \operatorname{Var}(Y) = \beta_1^2 + \beta_2^2 + 2\rho\beta_1\beta_2$. The Shapley effects are then $\tfrac12[c(\{1\}) + \operatorname{Var}(Y) - c(\{2\})]$ and $\tfrac12[c(\{2\}) + \operatorname{Var}(Y) - c(\{1\})]$. With $\rho = 0$ they reduce to $\beta_1^2$ and $\beta_2^2$, the Sobol indices times the variance.

In [ ]:
b1, b2, r = 2.0, 1.0, 0.6
V = b1**2 + b2**2 + 2 * r * b1 * b2
toy = {(): 0.0, (0,): b1**2 * (1 - r**2), (1,): b2**2 * (1 - r**2), (0, 1): V}
sh = shapley(toy, 2)
assert np.allclose(sh, [0.5 * (toy[(0,)] + V - toy[(1,)]), 0.5 * (toy[(1,)] + V - toy[(0,)])])
assert np.isclose(sh.sum(), V)
print("toy Shapley shares:", (sh / V).round(3), " passes")

### Run it on the emulator

There are $2^{11} = 2{,}048$ sets of groups. The cell below computes every cost and then the effects. With $N_O = 200$ and $N_I = 3$ this is about 1.2 million emulator predictions. It took under a minute on a laptop CPU; if yours is much slower, test the code with `N_OUTER = 50` first.

In [ ]:
N_OUTER, N_INNER = 200, 3
t0 = time.time()
costs = {}
for r in range(d + 1):
    for u in itertools.combinations(range(d), r):
        costs[u] = cost(u, N_OUTER, N_INNER)
    print(f"sets of size {r} done, {time.time() - t0:.0f} s")
SH = shapley(costs, d) / VAR_Y                                        # (11 groups, 3 joints, 101 gait points)
total = SH.sum(axis=0)
print(f"sum of the effects over the groups: between {total.min():.3f} and {total.max():.3f} (must be 1)")

**Check the Monte Carlo error.** Run the two cells above a second time with a different seed (`rng = np.random.default_rng(7)`), keep the first result as `SH_1`, and compare. Report the largest absolute difference. If it is above about 0.05, double `N_OUTER` and report how long the run took.

## Step 7. Variance-renormalized effects and figures

A large share of a tiny variance does not matter much. Benemerito et al. (2022) therefore weight the indices by the variance of the force at each gait point (their VRSI). Their printed formula divides by a sum over the inputs, which as written cancels the variance; we use the reading that matches their intent and the 0.1 threshold, and divide by the largest variance over the cycle:
$$
\mathrm{VRSI}_{ij}(t) = \mathrm{Sh}_{ij}(t)\, \frac{\operatorname{Var}(Y_j(t))}{\max_s \operatorname{Var}(Y_j(s))}.
$$
A group counts as influential at $t$ when $\mathrm{VRSI} \ge 0.1$, as in the paper.

**TODO.** Compute `VRSI` with the same shape as `SH`.

In [ ]:
VRSI = None   # TODO

In [ ]:
def heatmaps(A, title, vmax=None):
    fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
    for j, ax in enumerate(axes):
        im = ax.imshow(A[:, j, :], aspect="auto", cmap="viridis", vmin=0, vmax=vmax or A.max(), extent=[0, 100, d - 0.5, -0.5])
        ax.axvline(62, color="white", lw=0.8, ls=":")
        ax.set_title(f"{JOINTS[j]} JCF", loc="left"); ax.set_xlabel("Gait cycle [%]")
    axes[0].set_yticks(range(d)); axes[0].set_yticklabels(GROUP_NAMES)
    fig.colorbar(im, ax=axes, shrink=0.8); fig.suptitle(title); plt.show()

heatmaps(SH, "Shapley effects", vmax=1)
heatmaps(VRSI, "variance-renormalized Shapley effects")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for j, c in enumerate(JOINTS):
    axes[0].plot(GRID, VAR_Y[j], label=c)
axes[0].set_title("Var(Y) of the emulated force [BW²]", loc="left"); axes[0].set_xlabel("Gait cycle [%]"); axes[0].legend(frameon=False)
share = (VRSI >= 0.1).mean(axis=2) * 100                              # % of the cycle above the threshold, (groups, joints)
order = np.argsort(-share.max(axis=1))
for j, c in enumerate(JOINTS):
    axes[1].barh(np.arange(d) + (j - 1) * 0.27, share[order, j], height=0.27, label=c)
axes[1].set_yticks(range(d)); axes[1].set_yticklabels([GROUP_NAMES[i] for i in order]); axes[1].invert_yaxis()
axes[1].set_xlabel("% of the gait cycle with VRSI ≥ 0.1"); axes[1].legend(frameon=False)
fig.tight_layout(); plt.show()
pd.DataFrame(share, index=GROUP_NAMES, columns=JOINTS).round(0)

## What to report

1. The table of $K_c$ from Step 1 and the figure of real and simulated curves from Step 4, with your comments.
2. The Monte Carlo check of Step 6: seeds, $N_O$, $N_I$, run time, largest difference.
3. The two heatmaps and the bar chart.
4. A short paragraph per joint: which inputs dominate in stance and in swing, and whether that makes biomechanical sense. For example, the ground reaction forces cannot explain much in swing, because they are zero there.
5. The limits of the analysis as you see them. Think about the normal model for speed, mass and height (four speeds, twelve body sizes), the 5 % of curve variance dropped by the PCA, and the fact that the emulator is a fitted model with its own error. Note also that the emulator uses one input range for all 811 inputs of a joint, so it cannot by itself decide that one curve matters more than another; the effects come from how each group moves the input vector and from the correlations between the groups. A model with one range per group would be the natural next step.

Save the arrays at the end so the results can be reused without rerunning:

In [ ]:
np.savez("shapley_results.npz", SH=SH, VRSI=VRSI, VAR_Y=VAR_Y, groups=np.array(GROUP_NAMES), n_outer=N_OUTER, n_inner=N_INNER)

## References

Benemerito, I., Montefiori, E., Marzo, A. and Mazzà, C. (2022). Reducing the complexity of musculoskeletal models using Gaussian process emulators. *Applied Sciences*, 12, 12932.

Owen, A. B. (2014). Sobol' indices and Shapley value. *SIAM/ASA Journal on Uncertainty Quantification*, 2, 245-251.

Song, E., Nelson, B. L. and Staum, J. (2016). Shapley effects for global sensitivity analysis: theory and computation. *SIAM/ASA Journal on Uncertainty Quantification*, 4, 1060-1083.

Iooss, B. and Prieur, C. (2019). Shapley effects for sensitivity analysis with correlated inputs: comparisons with Sobol' indices, numerical estimation and applications. *International Journal for Uncertainty Quantification*, 9, 493-514.